# Perceptron Pipeline on MNIST

This notebook builds an end-to-end machine learning pipeline using a **Perceptron** model for the MNIST digit classification dataset.

Tasks covered:
- Load the MNIST dataset using Pandas
- Split into training and test sets
- Build a Scikit-Learn pipeline with preprocessing and a Perceptron model
- Tune important hyperparameters using cross-validation
- Evaluate the best model on the test set
- Inspect important model attributes and methods


In [ ]:
# 1. Import required libraries

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Perceptron
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 2. Load the MNIST dataset

The teaching notebook asks us to load the MNIST dataset using Pandas.  
This code first tries common local paths. If your lecturer's dataset has a different name, update the `possible_paths` list below.


In [ ]:
# 2. Load data using Pandas

possible_paths = [
    "datasets/mnist.csv",
    "../datasets/mnist.csv",
    "../../datasets/mnist.csv",
    "mnist.csv"
]

data_path = None
for path in possible_paths:
    if os.path.exists(path):
        data_path = path
        break

if data_path is None:
    raise FileNotFoundError(
        "MNIST CSV file not found. Please place mnist.csv inside a datasets folder "
        "or update the possible_paths list with the correct file path."
    )

mnist = pd.read_csv(data_path)

print("Dataset loaded from:", data_path)
print("Shape:", mnist.shape)
mnist.head()


## 3. Basic data understanding

MNIST normally contains pixel columns and one target label column.  
The target column is commonly called `label`, `target`, `class`, or may be the first/last column.


In [ ]:
# 3. Identify target column

print("Columns:")
print(mnist.columns[:10].tolist(), "...")

candidate_targets = ["label", "target", "class", "digit", "y"]

target_col = None
for col in candidate_targets:
    if col in mnist.columns:
        target_col = col
        break

if target_col is None:
    # Common MNIST CSV format: label is the first column
    target_col = mnist.columns[0]

print("Target column selected:", target_col)

X = mnist.drop(columns=[target_col])
y = mnist[target_col]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("Target classes:", sorted(y.unique())[:20])


In [ ]:
# 4. Check missing values and class distribution

print("Missing values:", mnist.isna().sum().sum())

class_counts = y.value_counts().sort_index()
display(class_counts)

plt.figure(figsize=(8, 4))
class_counts.plot(kind="bar")
plt.title("Class Distribution of MNIST Digits")
plt.xlabel("Digit")
plt.ylabel("Number of Samples")
plt.show()


## 4. Visualise sample images

Each MNIST image is 28 × 28 pixels. The code below displays a few examples.


In [ ]:
# 5. Display sample digit images

def show_digit(row, label):
    image = row.to_numpy().reshape(28, 28)
    plt.imshow(image, cmap="gray")
    plt.title(f"Label: {label}")
    plt.axis("off")

plt.figure(figsize=(10, 4))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    show_digit(X.iloc[i], y.iloc[i])
plt.tight_layout()
plt.show()


## 5. Train-test split

The data is split into training and test sets.  
`stratify=y` keeps the digit proportions similar in both sets.


In [ ]:
# 6. Split dataset

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)


## 6. Build a baseline Perceptron pipeline

The pipeline includes:
1. **StandardScaler** to scale pixel values.
2. **Perceptron** as the classification model.


In [ ]:
# 7. Baseline pipeline

baseline_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("perceptron", Perceptron(random_state=RANDOM_STATE))
])

baseline_pipe.fit(X_train, y_train)

baseline_pred = baseline_pipe.predict(X_test)

baseline_accuracy = accuracy_score(y_test, baseline_pred)
baseline_f1 = f1_score(y_test, baseline_pred, average="weighted")

print("Baseline Accuracy:", round(baseline_accuracy, 4))
print("Baseline Weighted F1:", round(baseline_f1, 4))


## 7. Hyperparameter tuning

Important Perceptron hyperparameters include:
- `penalty`: regularisation type
- `alpha`: regularisation strength
- `max_iter`: maximum number of training iterations
- `eta0`: learning rate
- `tol`: stopping tolerance


In [ ]:
# 8. GridSearchCV for Perceptron

param_grid = {
    "perceptron__penalty": [None, "l2", "l1"],
    "perceptron__alpha": [0.0001, 0.001, 0.01],
    "perceptron__max_iter": [1000, 2000],
    "perceptron__eta0": [0.1, 1.0],
    "perceptron__tol": [1e-3]
}

search = GridSearchCV(
    estimator=baseline_pipe,
    param_grid=param_grid,
    scoring="f1_weighted",
    cv=3,
    n_jobs=-1,
    return_train_score=True
)

search.fit(X_train, y_train)

print("Best CV score:", round(search.best_score_, 4))
print("Best parameters:")
search.best_params_


In [ ]:
# 9. View top tuning results

results = pd.DataFrame(search.cv_results_)
results_table = results[
    [
        "mean_test_score",
        "mean_train_score",
        "param_perceptron__penalty",
        "param_perceptron__alpha",
        "param_perceptron__max_iter",
        "param_perceptron__eta0"
    ]
].sort_values("mean_test_score", ascending=False)

results_table.head(10)


## 8. Test the best model

After selecting the best pipeline using cross-validation, the final evaluation is performed on the unseen test set.


In [ ]:
# 10. Evaluate best model on test set

best_model = search.best_estimator_
test_pred = best_model.predict(X_test)

accuracy = accuracy_score(y_test, test_pred)
precision = precision_score(y_test, test_pred, average="weighted")
recall = recall_score(y_test, test_pred, average="weighted")
f1 = f1_score(y_test, test_pred, average="weighted")

metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision (weighted)", "Recall (weighted)", "F1-score (weighted)"],
    "Score": [accuracy, precision, recall, f1]
})

metrics["Score"] = metrics["Score"].round(4)
metrics


In [ ]:
# 11. Full classification report

print(classification_report(y_test, test_pred))


In [ ]:
# 12. Confusion matrix

cm = confusion_matrix(y_test, test_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(values_format="d")
plt.title("Confusion Matrix - Perceptron on MNIST")
plt.show()


## 9. Inspect important attributes and methods

The trained Perceptron stores important learned information:
- `coef_`: learned weights
- `intercept_`: bias terms
- `classes_`: class labels
- `n_iter_`: number of iterations used during training


In [ ]:
# 13. Inspect Perceptron attributes

trained_perceptron = best_model.named_steps["perceptron"]

print("Classes:", trained_perceptron.classes_)
print("Coefficient shape:", trained_perceptron.coef_.shape)
print("Intercept shape:", trained_perceptron.intercept_.shape)
print("Number of iterations:", trained_perceptron.n_iter_)


In [ ]:
# 14. Use important methods in practice

sample = X_test.iloc[:5]

predictions = best_model.predict(sample)
decision_scores = best_model.decision_function(sample)

print("Predicted labels:", predictions)
print("Decision function shape:", decision_scores.shape)


## 10. Conclusion

This notebook implemented an end-to-end Perceptron classification pipeline for the MNIST dataset.  
The pipeline used scaling, model training, hyperparameter tuning, and final evaluation on a separate test set.

The Perceptron is a simple linear classifier. It can perform reasonably well on MNIST, but because it only learns linear decision boundaries, more advanced models such as multi-layer neural networks can usually achieve better performance on complex image classification tasks.
